# CRDT thermostat over iroh
A service whose fields are CRDTs (`effect-replicated`), replicated over a private iroh network (`effect-iroh`). Element code only uses atom get/set. Run from Tern; the first cell registers `%%ts`.

In [1]:
# Registers %%ts in Tern's kernel: runs the cell as TypeScript under Bun, from this folder.
# Tern's kernel has no public magic registry (get_ipython() is None), so find its Magics instance.
import gc, os, shutil, subprocess
import tern_kernel.magics as tern_magics

BUN = shutil.which("bun") or os.path.expanduser("~/.bun/bin/bun")  # the kernel's PATH may lack Bun

def ts(line, cell):
    """Runs the cell as TypeScript with Bun, in the kernel's working directory."""
    p = subprocess.Popen([BUN, "run", "-"], cwd=os.getcwd(), text=True,
                         stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    p.stdin.write(cell); p.stdin.close()
    for out in p.stdout:
        print(out, end="")
    if p.wait() != 0:
        raise RuntimeError(f"bun exited {p.returncode}")

magics = next(o for o in gc.get_objects() if isinstance(o, tern_magics.Magics))
magics._cell["ts"] = ts
print("cwd:", os.getcwd())

cwd: /Users/eshingriffith/ghq/github.com/egriff38/effect-iroh/examples/crdt-thermostat


## 1. Resolver laws (no network)
Merge is commutative, associative and idempotent, so any order, including duplicates, gives the same state.

In [2]:
%%ts
// The resolver laws, no network: merge is order-independent, so replicas converge.
import { Schema } from "effect"
import { Crdt } from "effect-replicated"

let [x, y, z] = [Crdt.PNCounter.empty, Crdt.PNCounter.empty, Crdt.PNCounter.empty]
for (let i = 0; i < 100; i++) {
  x = Crdt.PNCounter.add(x, "X", 1); y = Crdt.PNCounter.add(y, "Y", 1); z = Crdt.PNCounter.add(z, "Z", 1)
}
const orders = [[x, y, z], [z, y, x], [y, x, z], [x, x, y, z, z]] // incl. duplicates
console.table(orders.map((o, i) => ({ order: i, value: Crdt.PNCounter.value(o.reduce(Crdt.PNCounter.merge)) })))

// Multi-value register: concurrent writes are kept; the resolver picks on read.
const Setpoint = Crdt.MVRegister(Schema.Number, (c) => Math.max(...c))
const merged = [Setpoint.set(Setpoint.empty, "X", 19), Setpoint.set(Setpoint.empty, "Y", 22), Setpoint.set(Setpoint.empty, "Z", 21)]
  .reduce(Setpoint.merge)
console.log("concurrent:", Setpoint.value(merged))
console.log("X writes 20 after seeing all:", Setpoint.value(Setpoint.merge(merged, Setpoint.set(merged, "X", 20))))

┌───┬───────┬───────┐
│   │ order │ value │
├───┼───────┼───────┤
│ 0 │ 0     │ 300   │
│ 1 │ 1     │ 300   │
│ 2 │ 2     │ 300   │
│ 3 │ 3     │ 300   │
└───┴───────┴───────┘
concurrent: {
  value: 22,
  candidates: [ 19, 22, 21 ],
}
X writes 20 after seeing all: {
  value: 20,
  candidates: [ 20 ],
}


## 2. Three replicas over iroh
The code is [src/main.ts](src/main.ts). X, Y and Z bind ephemeral ports in one process; each adds 1 to `energy` 100 times, then all three set conflicting setpoints at once; X then writes 20.

In [3]:
%%ts
await import("./src/main.ts")

X 2276cbd94c @ 127.0.0.1:53777 [::]:59884
Y 6526bae486 @ 127.0.0.1:52402 [::]:58040
Z f78390bc2c @ 127.0.0.1:53974 [::]:55537
┌───┬───────────────────┬─────────┬────────┬──────────┬────────────┐
│   │ step              │ replica │ energy │ setpoint │ candidates │
├───┼───────────────────┼─────────┼────────┼──────────┼────────────┤
│ 0 │ concurrent writes │ X       │ 300    │ 22       │ 19, 22, 21 │
│ 1 │ concurrent writes │ Y       │ 300    │ 22       │ 19, 22, 21 │
│ 2 │ concurrent writes │ Z       │ 300    │ 22       │ 19, 22, 21 │
│ 3 │ X writes 20       │ X       │ 300    │ 20       │ 20         │
│ 4 │ X writes 20       │ Y       │ 300    │ 20       │ 20         │
│ 5 │ X writes 20       │ Z       │ 300    │ 20       │ 20         │
└───┴───────────────────┴─────────┴────────┴──────────┴────────────┘
